# Reason audit: what does the model rely on, and is it stable?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/reason-audit.ipynb)

Companion notebook for the ReliableML paper `docs/latex/reason-audit.tex`. It uses `audit_model_reasons` (or `ReliablePipeline.audit_reasons`): predictive validity with bootstrap intervals, perturbation-based feature reliance, slice stability, counterfactual sensitivity, logistic association and explanation alignment.

All numbers shown below are computed by the cells themselves (fixed random seeds); nothing is hard-coded. Run the cells top to bottom.

## 0. Setup (parameters)

**Parameter cell - choose ONE installation source.** The default installs the released package from PyPI. To try the unreleased development version, comment out option A and uncomment option B. If neither works (for example a restricted network), run the fallback cell below it and upload a zip of the repository.

In [ ]:
#@title PARAMETERS: install ReliableML
# ---- Option A (default): released version from PyPI ----
!pip install -q ReliableML
# ---- Option B: unreleased version from GitHub (uncomment; comment option A) ----
# !pip install -q "git+https://github.com/Jabualani/ML-Reliabilty.git"
# Plots need matplotlib (preinstalled on Colab); locally: pip install matplotlib

In [ ]:
#@title Fallback: upload a repository zip (only runs if ml_reliability is not importable)
import importlib.util, pathlib, sys, zipfile

if importlib.util.find_spec("ml_reliability") is None:
    from google.colab import files  # Colab only

    # GitHub: Code > Download ZIP, then pick the file in the dialog
    uploaded = files.upload()
    for archive in uploaded:
        zipfile.ZipFile(archive).extractall("repo_zip")
    package_init = next(pathlib.Path("repo_zip").rglob("ml_reliability/__init__.py"))
    sys.path.insert(0, str(package_init.parent.parent))
    # the repo's dependencies (numpy, pandas, scikit-learn, scipy, statsmodels) are
    # preinstalled on Colab; otherwise: !pip install -q numpy pandas scikit-learn scipy statsmodels

import ml_reliability
print("ml_reliability loaded from:", pathlib.Path(ml_reliability.__file__).parent)

## 1. Quantities

**Predictive validity** on a labeled holdout $D$ ($n$ rows, probabilities $p_i$, one-hot $e_{y_i}$):
$$B=\frac1n\sum_{i=1}^n\lVert p_i-e_{y_i}\rVert_2^2,\qquad \operatorname{ECE}=\sum_{b=1}^{10}\frac{\lvert I_b\rvert}{n}\,\bigl\lvert\operatorname{acc}(I_b)-\operatorname{conf}(I_b)\bigr\rvert,$$
with row-bootstrap intervals for accuracy, Brier score and ECE.

**Perturbation-based feature reliance.** With multiclass Brier loss $L(f;D)$ and $D^{(j)}$ the data in which feature $j$ is replaced by donor values from other audit rows (optionally nearest donors on `conditional_on` features),
$$\Delta_j=\mathbb E\bigl[L(f;D^{(j)})-L(f;D)\bigr].$$
A positive *lower* interval bound is evidence of reliance **under this perturbation procedure** - not a causal effect, nor necessity in the data-generating process. **Negative controls** (`negative_control_features`) should show no reliance; **unavailable features** (`unavailable_features`, not known at prediction time) with positive reliance are leakage/shortcut indicators.

**Stability** repeats the reliance calculation within `groups` and `time_periods` slices. **Counterfactual sensitivity** replaces a column by user-supplied values and reports the prediction-flip rate and $\overline{\lvert\Delta\hat p_+\rvert}$ with row-bootstrap intervals. **Logistic association** fits an unregularized logistic model on standardized features and reports odds ratios per standard deviation with Benjamini-Hochberg adjusted $p$-values - a descriptive summary of associations in the audit data, not a model explanation. Explanation alignment compares SHAP-like and LIME-like attributions (see the explainability notebook).

## 2. Synthetic data with a planted shortcut

Features: `signal` and `weak` truly drive the label; `noise` and `negctl` do not; `leak` is a noisy copy of the label that would not be available at prediction time. The model sees all of them.

In [ ]:
import warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from ml_reliability import ReliablePipeline, audit_model_reasons

warnings.filterwarnings("ignore")
rng = np.random.default_rng(0)
n = 900
X = pd.DataFrame(rng.normal(size=(n, 4)), columns=["signal", "weak", "noise", "negctl"])
y = (rng.random(n) < 1 / (1 + np.exp(-(1.5 * X["signal"] + 0.5 * X["weak"])))).astype(int)
X["leak"] = y + rng.normal(scale=0.3, size=n)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.35, random_state=0)
forest = RandomForestClassifier(n_estimators=25, random_state=0).fit(Xtr, ytr)

group = np.where(Xte["signal"] > 0, "signal>0", "signal<=0")
period = np.array(["period_1", "period_2", "period_3"])[np.arange(len(Xte)) % 3]  # illustrative split
audit = audit_model_reasons(
    forest, Xte, yte, positive_label=1, background_data=Xtr,
    groups={"sign_of_signal": group}, time_periods=period,
    counterfactuals={"signal": np.zeros(len(Xte))},
    negative_control_features=("negctl",), unavailable_features=("leak",),
    bootstrap_samples=100, explanation_samples=100, max_explained_instances=15, random_seed=0,
)
pv = audit.predictive_validity
print(f"n={audit.sample_count}  accuracy={pv['accuracy']:.3f} {tuple(round(v, 3) for v in pv['accuracy_confidence_interval'])}"
      f"  Brier={pv['brier_score']:.3f} {tuple(round(v, 3) for v in pv['brier_score_confidence_interval'])}"
      f"  ECE={pv['top_label_expected_calibration_error']:.3f} {tuple(round(v, 3) for v in pv['expected_calibration_error_confidence_interval'])}")

In [ ]:
reliance = pd.DataFrame([{
    "feature": r["feature"], "loss_increase": r["loss_increase"],
    "CI low": r["loss_increase_confidence_interval"][0], "CI high": r["loss_increase_confidence_interval"][1],
    "lower bound > 0": r["positive_lower_bound"],
    "negative control relied on": r.get("negative_control_warning", ""),
    "unavailable feature relied on": r.get("unavailable_at_prediction_time_warning", ""),
} for r in audit.feature_reliance])
print(reliance.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 3.6))
order = reliance.sort_values("loss_increase")
ax.barh(order["feature"], order["loss_increase"], xerr=[order["loss_increase"] - order["CI low"], order["CI high"] - order["loss_increase"]], capsize=3)
ax.axvline(0, color="k", lw=0.6); ax.set_xlabel("Brier-loss increase when the feature is perturbed"); ax.set_title("Feature reliance with 95% bootstrap CI")
plt.tight_layout(); plt.show()

In [ ]:
print("Stability across slices (loss increase, lower CI bound > 0 marked with *):")
rows = []
for kind, slices in [("group", audit.stability["groups"]["sign_of_signal"]), ("period", audit.stability["time_periods"]["time_period"])]:
    for item in slices:
        row = {"slice": f"{kind}:{item['value']}", "n": item["sample_count"]}
        for feature, stats_ in item["feature_reliance"].items():
            low = stats_["loss_increase_confidence_interval"][0]
            row[feature] = f"{stats_['loss_increase']:.3f}{'*' if low > 0 else ''}"
        rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))
print("between-slice SD of reliance:", {k: round(v["between_slice_standard_deviation"], 4) for k, v in audit.stability["features"].items()})
cf = audit.counterfactual_sensitivity["features"]["signal"]
print("Counterfactual signal:=0 -> flip rate", round(cf["prediction_flip_rate"], 3), tuple(round(v, 3) for v in cf["prediction_flip_rate_confidence_interval"]),
      "| mean |change in P(Y=1)|", round(cf["mean_absolute_positive_probability_change"], 3))
print()
odds = pd.DataFrame(audit.logistic_association["odds_ratios"]).T
print(odds[["odds_ratio_per_one_standard_deviation", "p_value", "fdr_bh_adjusted_p_value"]].round(4).to_string())
print("\nexplanation alignment:", {k: v for k, v in audit.explanation_alignment.items() if k in ["status", "alignment_score_0_to_1", "mean_absolute_importance_spearman"]})
print("\nwarnings:")
for w in audit.warnings:
    print(" -", w)

## 3. Real data through the pipeline: breast cancer

`ReliablePipeline.audit_reasons` reuses the fitted model and the training data as background. The groups and time periods below are **illustrative**: tumour-size groups and row-order periods, not real protected groups or real time. Features are chosen to avoid near-duplicates (radius, perimeter and area are almost collinear, which makes odds ratios unstable).

In [ ]:
bc = load_breast_cancer(as_frame=True)
cols = ["mean radius", "mean texture", "mean smoothness", "mean compactness", "mean symmetry", "mean fractal dimension"]
Xb, yb = bc.data[cols], bc.target
Xbtr, Xbte, ybtr, ybte = train_test_split(Xb, yb, test_size=0.35, random_state=0, stratify=yb)
pipe = ReliablePipeline(LogisticRegression(max_iter=5000), plot_dir="reliability_plots", random_seed=0).fit(Xbtr, ybtr)
size = np.where(Xbte["mean radius"] > Xbte["mean radius"].median(), "large", "small")
period_b = np.array(["early", "late"])[(np.arange(len(Xbte)) >= len(Xbte) // 2).astype(int)]
bc_audit = pipe.audit_reasons(
    Xbte, ybte, positive_label=1, groups={"size": size}, time_periods=period_b,
    counterfactuals={"mean texture": np.full(len(Xbte), Xbtr["mean texture"].median())},
    conditional_on=["mean radius"], bootstrap_samples=100, explanation_samples=100, max_explained_instances=15,
)
pv = bc_audit.predictive_validity
print(f"accuracy={pv['accuracy']:.3f}  Brier={pv['brier_score']:.3f}  ECE={pv['top_label_expected_calibration_error']:.3f}")
print(pd.DataFrame([{"feature": r["feature"], "loss_increase": r["loss_increase"], "CI": tuple(round(v, 4) for v in r["loss_increase_confidence_interval"]),
                     "lower>0": r["positive_lower_bound"]} for r in bc_audit.feature_reliance]).round(4).to_string(index=False))
print(pd.DataFrame(bc_audit.logistic_association["odds_ratios"]).T[["odds_ratio_per_one_standard_deviation", "fdr_bh_adjusted_p_value"]].round(4).to_string())
print("alignment:", {k: v for k, v in bc_audit.explanation_alignment.items() if k in ["status", "alignment_score_0_to_1"]})

## Interpretation and limitations

**How to read the output.** A positive lower bound in the reliance table says the model's loss rises when that feature is scrambled, i.e. the model uses it. The planted `leak` feature dominates and is flagged because it was declared unavailable at prediction time; the negative control and `noise` should have intervals that include zero. Slice tables show whether reliance is stable across groups and periods (small slices are noisy). Counterfactual rows test the model, not the world.

**Limitations.** Reliance is relative to the perturbation scheme (marginal donors create unrealistic rows when features are correlated; conditioning only approximates the conditional distribution); it is not causal and does not identify necessity. Odds ratios are descriptive associations that become unstable under collinearity or separation; intervals assume independent rows and no multiplicity control beyond the reported BH adjustment; the auditor must declare which features are unavailable or negative controls; the audit data must be representative holdout data not used for training.

These notebooks are methodological demonstrations on synthetic data and small bundled scikit-learn datasets. They do not show that any method improves performance on your data; that requires a prespecified evaluation on independent data.